# Analiza porownawcza KAN vs MLP w zadaniach regresji

**Praca magisterska** - Kolmogorov-Arnold Networks w zastosowaniach regresyjnych
> Wersja: **macOS / Apple Silicon (MPS)**


## Spis tresci
1. Konfiguracja srodowiska i diagnostyka GPU/CPU
2. Klasy pomocnicze (datasety, modele)
3. Liczba parametrow - analiza teoretyczna i empiryczna
4. Hyperparameter Tuning KAN
5. Hyperparameter Tuning MLP
6. Eksperymenty na funkcjach syntetycznych (1D)
7. Eksperymenty na funkcjach wielowymiarowych (2D)
8. Eksperymenty na zbiorach rzeczywistych (Diabetes)
9. Wplyw grid refinement na dokladnosc KAN
10. Analiza czasu treningu i efektywnosci
11. Interpretowalnosc KAN (wizualizacja struktury)
12. Pruning sieci KAN
13. Symbolic fitting - dopasowanie symboliczne
14. Feature attribution - waznosc cech (Diabetes)
15. Metryki regularyzacji (reg_metric)
16. Odpornosc na szum i efektywnosc probkowa
17. Podsumowanie i wnioski

## 1. Konfiguracja srodowiska i diagnostyka GPU/CPU

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from itertools import product
import torch
import torch.nn as nn
import time
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
import keras
from kan import KAN, create_dataset
from sympy import *

plt.style.use("./deeplearning.mplstyle")

# Priorytet: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)

print("=" * 60)
print("DIAGNOSTYKA SRODOWISKA OBLICZENIOWEGO")
print("=" * 60)
print(f"PyTorch: {torch.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"NumPy: {np.__version__}")
import platform
print(f"System: {platform.system()} {platform.machine()}")
print()

# PyTorch GPU
if torch.cuda.is_available():
    print(f"[PyTorch] CUDA GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"  VRAM: {props.total_mem / 1024**3:.1f} GB")
    print(f"  CUDA: {torch.version.cuda}")
elif torch.backends.mps.is_available():
    print("[PyTorch] Apple Silicon GPU (MPS) dostepne")
    print(f"  MPS built: {torch.backends.mps.is_built()}")
    import subprocess
    try:
        mem = subprocess.check_output(["sysctl", "-n", "hw.memsize"]).decode().strip()
        print(f"  Unified Memory: {int(mem) / 1024**3:.0f} GB")
    except: pass
    try:
        chip = subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"]).decode().strip()
        print(f"  Procesor: {chip}")
    except: pass
else:
    print("[PyTorch] GPU: niedostepne, uzyje CPU")

# TensorFlow GPU
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"\n[TensorFlow] GPU: {len(gpus)} device(s)")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("\n[TensorFlow] GPU: niedostepne")
    if platform.machine() == "arm64":
        print("  TIP: pip install tensorflow-metal")

print(f"\nDEVICE = \"{DEVICE}\" (KAN/PyTorch)")
if DEVICE == "mps":
    print("Apple Silicon MPS - GPU acceleration aktywne")
print("Caly kod jest uniwersalny - automatycznie wykrywa i uzywa GPU jesli dostepne.")

## 2. Klasy pomocnicze

In [ ]:
class KANDatasets:
    """Dataset utility class. Dane automatycznie na DEVICE."""

    @staticmethod
    def _to_pykan_format(X_train, X_test, y_train, y_test, device=DEVICE):
        return {
            "train_input": torch.tensor(X_train, dtype=torch.float32).to(device),
            "train_label": torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1).to(device),
            "test_input": torch.tensor(X_test, dtype=torch.float32).to(device),
            "test_label": torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1).to(device)
        }

    @staticmethod
    def create_polynomial_dataset(n_samples=500, degree=3, noise=0.1, test_split=0.2, seed=SEED):
        """y = x + x^2 + x^3 + noise"""
        np.random.seed(seed)
        x = np.linspace(-2, 2, n_samples)
        y = sum(x**i for i in range(1, degree + 1)) + np.random.normal(0, noise, x.shape)
        X = x.reshape(-1, 1)
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_split, random_state=seed)
        return KANDatasets._to_pykan_format(Xtr, Xte, ytr, yte)

    @staticmethod
    def create_sinusoidal_dataset(n_samples=500, noise=0.1, test_split=0.2, seed=SEED):
        """y = sin(2*pi*x) + noise"""
        np.random.seed(seed)
        x = np.linspace(-2, 2, n_samples)
        y = np.sin(2 * np.pi * x) + np.random.normal(0, noise, x.shape)
        X = x.reshape(-1, 1)
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_split, random_state=seed)
        return KANDatasets._to_pykan_format(Xtr, Xte, ytr, yte)

    @staticmethod
    def create_complex_dataset(n_samples=500, noise=0.1, test_split=0.2, seed=SEED):
        """y = sin(x)*cos(3x) + exp(-0.1*x^2) + noise"""
        np.random.seed(seed)
        x = np.linspace(-3, 3, n_samples)
        y = np.sin(x) * np.cos(3*x) + np.exp(-0.1 * x**2) + np.random.normal(0, noise, x.shape)
        X = x.reshape(-1, 1)
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_split, random_state=seed)
        return KANDatasets._to_pykan_format(Xtr, Xte, ytr, yte)

    @staticmethod
    def create_2d_dataset(n_samples=1000, noise=0.1, test_split=0.2, seed=SEED):
        """y = exp(sin(pi*x1) + x2^2) + noise"""
        np.random.seed(seed)
        X = np.random.uniform(-1, 1, (n_samples, 2))
        y = np.exp(np.sin(np.pi * X[:, 0]) + X[:, 1]**2) + np.random.normal(0, noise, n_samples)
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_split, random_state=seed)
        return KANDatasets._to_pykan_format(Xtr, Xte, ytr, yte)

    @staticmethod
    def create_diabetes_dataset(test_split=0.2, seed=SEED):
        """Sklearn Diabetes (442 samples, 10 features)"""
        X, y = load_diabetes(return_X_y=True)
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=test_split, random_state=seed)
        return KANDatasets._to_pykan_format(Xtr, Xte, ytr, yte)

print(f"KANDatasets zaladowane. Dane na: {DEVICE}")

In [ ]:
class KANRegressor:
    """KAN wrapper. Automatycznie GPU."""

    def __init__(self, width=[2, 5, 1], grid=5, k=3, seed=SEED, device=DEVICE):
        self.width = list(width)
        self.grid = grid
        self.k = k
        self.seed = seed
        self.device = device
        self.model = None
        self.train_time = 0
        self.results = None

    def fit(self, dataset, opt="LBFGS", steps=50, lamb=0.001, lamb_entropy=2.0,
            reg_metric="edge_forward_spline_n"):
        input_dim = dataset["train_input"].shape[1]
        if self.width[0] != input_dim:
            self.width[0] = input_dim
        self.model = KAN(width=self.width, grid=self.grid, k=self.k,
                         seed=self.seed, device=self.device)
        start = time.time()
        self.results = self.model.fit(dataset, opt=opt, steps=steps,
                                      lamb=lamb, lamb_entropy=lamb_entropy,
                                      reg_metric=reg_metric)
        self.train_time = time.time() - start
        return self

    def predict(self, X):
        if isinstance(X, np.ndarray):
            X = torch.tensor(X, dtype=torch.float32)
        X = X.to(self.device)
        self.model.eval()
        with torch.no_grad():
            return self.model(X).cpu().numpy()

    def count_parameters(self):
        if self.model is None:
            return sum(self.width[i]*self.width[i+1]*(self.grid+self.k+1)
                       for i in range(len(self.width)-1))
        return sum(p.numel() for p in self.model.parameters())

    def evaluate(self, dataset):
        X_test = dataset["test_input"].cpu().numpy()
        y_test = dataset["test_label"].cpu().numpy().flatten()
        y_pred = self.predict(X_test).flatten()
        return {"mse": mean_squared_error(y_test, y_pred),
                "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
                "mae": mean_absolute_error(y_test, y_pred),
                "r2": r2_score(y_test, y_pred)}

print(f"KANRegressor zaladowany. Device: {DEVICE}")

In [ ]:
class MLPRegressor:
    """MLP (TensorFlow/Keras). GPU automatycznie."""

    def __init__(self, hidden_layers=[64,32], activation="relu",
                 epochs=200, learning_rate=0.001, seed=SEED):
        self.hidden_layers = hidden_layers
        self.activation = activation
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.seed = seed
        self.model = None
        self.history = None
        self.train_time = 0
        self.scaler = StandardScaler()

    def _build_model(self, input_dim):
        tf.random.set_seed(self.seed)
        model = keras.Sequential()
        model.add(keras.layers.Input(shape=(input_dim,)))
        for units in self.hidden_layers:
            model.add(keras.layers.Dense(units, activation=self.activation))
        model.add(keras.layers.Dense(1, activation="linear"))
        model.compile(optimizer=keras.optimizers.Adam(
            learning_rate=self.learning_rate), loss="mse")
        return model

    def fit(self, dataset):
        X_train = dataset["train_input"].cpu().numpy()
        y_train = dataset["train_label"].cpu().numpy()
        X_val = dataset["test_input"].cpu().numpy()
        y_val = dataset["test_label"].cpu().numpy()
        X_train = self.scaler.fit_transform(X_train)
        X_val = self.scaler.transform(X_val)
        self.model = self._build_model(X_train.shape[1])
        start = time.time()
        self.history = self.model.fit(X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=self.epochs, batch_size=32, verbose=0)
        self.train_time = time.time() - start
        return self

    def predict(self, X):
        if hasattr(X, "cpu"): X = X.cpu().numpy()
        X = self.scaler.transform(X)
        return self.model.predict(X, verbose=0)

    def count_parameters(self):
        return self.model.count_params() if self.model else 0

    def evaluate(self, dataset):
        X_test = dataset["test_input"].cpu().numpy()
        y_test = dataset["test_label"].cpu().numpy().flatten()
        y_pred = self.predict(X_test).flatten()
        return {"mse": mean_squared_error(y_test, y_pred),
                "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
                "mae": mean_absolute_error(y_test, y_pred),
                "r2": r2_score(y_test, y_pred)}

print("MLPRegressor (TensorFlow/Keras) zaladowany.")

## 3. Liczba parametrow

**KAN:** $P = \sum n_l \cdot n_{l+1} \cdot (G+k+1)$ &nbsp;&nbsp; **MLP:** $P = \sum (n_l \cdot n_{l+1} + n_{l+1})$

In [ ]:
def count_kan_params(width, grid, k):
    return sum(width[i]*width[i+1]*(grid+k+1) for i in range(len(width)-1))

def count_mlp_params(width):
    return sum(width[i]*width[i+1]+width[i+1] for i in range(len(width)-1))

print("=" * 70)
cfgs = [
    ("Mala 1D","KAN",[1,3,1],5,3), ("Mala 1D","MLP",[1,32,16,1],0,0),
    ("Srednia 1D","KAN",[1,5,3,1],5,3), ("Srednia 1D","MLP",[1,64,32,1],0,0),
    ("Duza 2D","KAN",[2,5,5,1],5,3), ("Duza 2D","MLP",[2,128,64,1],0,0),
    ("Diabetes 10D","KAN",[10,8,4,1],5,3), ("Diabetes 10D","MLP",[10,128,64,32,1],0,0),
]
for n, t, w, g, k in cfgs:
    p = count_kan_params(w,g,k) if t=="KAN" else count_mlp_params(w)
    info = f"G={g},k={k}" if t=="KAN" else "-"
    print(f"{n:<20} {t:<5} {info:<10} {p:>8,}")
print("=" * 70)

In [ ]:
grids = [3, 5, 8, 10, 15, 20, 30, 50]
kan_p = [count_kan_params([1,3,1], g, 3) for g in grids]
mlp_w = [[1,8,1],[1,12,1],[1,16,1],[1,20,1],[1,28,1],[1,38,1],[1,56,1],[1,92,1]]
mlp_p = [count_mlp_params(w) for w in mlp_w]
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(grids, kan_p, "o-", label="KAN [1,3,1]", markersize=8)
ax.plot(grids, mlp_p, "s-", label="MLP [1,N,1]", markersize=8)
ax.set_xlabel("Grid / Neurony"); ax.set_ylabel("Parametry")
ax.set_title("Skalowanie parametrow"); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("./figures/param_scaling.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Hyperparameter Tuning KAN

Grid search po: **width**, **grid**, **lamb**, **k**

In [ ]:
dataset_hp = KANDatasets.create_sinusoidal_dataset(n_samples=500, noise=0.1)

widths = [[1,3,1], [1,5,1], [1,5,3,1], [1,3,3,1], [1,8,1]]
grid_vals = [3, 5, 8, 10, 15]
lamb_vals = [0.0, 0.001, 0.01, 0.1]

print(f"KAN Grid Search: {len(widths)*len(grid_vals)*len(lamb_vals)} kombinacji")
print("=" * 60)

hp_kan = []
for width in widths:
    for grid in grid_vals:
        for lamb in lamb_vals:
            try:
                m = KANRegressor(width=width, grid=grid, k=3)
                m.fit(dataset_hp, steps=30, lamb=lamb, lamb_entropy=2.0)
                ev = m.evaluate(dataset_hp)
                hp_kan.append({"width": str(width), "grid": grid, "lamb": lamb,
                    "r2": ev["r2"], "mse": ev["mse"],
                    "params": m.count_parameters(), "time": m.train_time})
            except:
                pass

df_hp_kan = pd.DataFrame(hp_kan).sort_values("r2", ascending=False)
print(f"Przeszukano {len(hp_kan)} konfiguracji.")
print("\nTop 10:")
print(df_hp_kan.head(10).to_string(index=False, float_format="%.4f"))
best_kan = df_hp_kan.iloc[0]
print(f"\nNAJLEPSZA: width={best_kan['width']}, grid={best_kan['grid']}, lamb={best_kan['lamb']}")
print(f"  R2={best_kan['r2']:.6f}, params={best_kan['params']:.0f}")

In [ ]:
# Wplyw stopnia splajnu k
print("Wplyw k (stopien splajnu):")
k_res = []
for k in [2, 3, 4, 5]:
    m = KANRegressor(width=[1,5,1], grid=8, k=k)
    m.fit(dataset_hp, steps=50, lamb=0.001)
    ev = m.evaluate(dataset_hp)
    k_res.append({"k": k, "r2": ev["r2"], "mse": ev["mse"],
                  "params": m.count_parameters()})
    print(f"  k={k}: R2={ev['r2']:.4f}, params={m.count_parameters()}")

df_k = pd.DataFrame(k_res)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.bar(df_k["k"], df_k["r2"])
a1.set_xlabel("k"); a1.set_ylabel("R2"); a1.set_title("R2 vs k")
a2.bar(df_k["k"], df_k["params"])
a2.set_xlabel("k"); a2.set_ylabel("Params"); a2.set_title("Params vs k")
plt.tight_layout()
plt.savefig("./figures/hp_spline_degree.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Heatmapa Grid vs Lambda
df_h = df_hp_kan[df_hp_kan["width"] == "[1, 5, 1]"].copy()
if len(df_h) > 0:
    piv = df_h.pivot_table(values="r2", index="grid", columns="lamb")
    plt.figure(figsize=(8, 6))
    sns.heatmap(piv, annot=True, fmt=".3f", cmap="YlOrRd")
    plt.xlabel("Lambda"); plt.ylabel("Grid size")
    plt.title("R2: Grid vs Lambda (KAN [1,5,1])")
    plt.tight_layout()
    plt.savefig("./figures/hp_kan_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()

## 5. Hyperparameter Tuning MLP (TensorFlow)

Grid search po: **hidden_layers**, **activation**, **learning_rate**

In [ ]:
mlp_hiddens = [[16], [32,16], [64,32], [64,32,16], [128,64,32]]
mlp_acts = ["relu", "tanh"]
mlp_lrs = [0.0001, 0.001, 0.01]

print(f"MLP Grid Search: {len(mlp_hiddens)*len(mlp_acts)*len(mlp_lrs)} kombinacji")
print("=" * 60)

hp_mlp = []
for hidden in mlp_hiddens:
    for act in mlp_acts:
        for lr in mlp_lrs:
            m = MLPRegressor(hidden_layers=hidden, activation=act,
                             learning_rate=lr, epochs=300)
            m.fit(dataset_hp)
            ev = m.evaluate(dataset_hp)
            hp_mlp.append({"hidden": str(hidden), "activation": act, "lr": lr,
                "r2": ev["r2"], "mse": ev["mse"],
                "params": m.count_parameters(), "time": m.train_time})

df_hp_mlp = pd.DataFrame(hp_mlp).sort_values("r2", ascending=False)
print(f"Przeszukano {len(hp_mlp)} konfiguracji MLP.")
print("\nTop 10:")
print(df_hp_mlp.head(10).to_string(index=False, float_format="%.4f"))
best_mlp = df_hp_mlp.iloc[0]
print(f"\nNAJLEPSZA: hidden={best_mlp['hidden']}, act={best_mlp['activation']}, lr={best_mlp['lr']}")
print(f"  R2={best_mlp['r2']:.6f}, params={best_mlp['params']:.0f}")

In [ ]:
print("\n" + "=" * 60)
print("POROWNANIE NAJLEPSZYCH (po tuning)")
print("=" * 60)
for label, kv, mv in [("R2", best_kan["r2"], best_mlp["r2"]),
                       ("MSE", best_kan["mse"], best_mlp["mse"]),
                       ("Params", best_kan["params"], best_mlp["params"]),
                       ("Czas[s]", best_kan["time"], best_mlp["time"])]:
    print(f"  {label:<12} KAN={kv:>12.4f}  MLP={mv:>12.4f}")
ratio = best_kan["params"] / best_mlp["params"]
print(f"\nStosunek parametrow KAN/MLP: {ratio:.2f}x")

## 6. Eksperymenty 1D

Wielomian, Sinus, Zlozona funkcja

In [ ]:
datasets_1d = {
    "Wielomian": KANDatasets.create_polynomial_dataset(noise=0.1),
    "Sinus": KANDatasets.create_sinusoidal_dataset(noise=0.1),
    "Zlozona": KANDatasets.create_complex_dataset(noise=0.1),
}
results_1d = []
for dn, ds in datasets_1d.items():
    print(f"\n{'='*50}\n{dn}\n{'='*50}")
    for w, g, nm in [([1,5,1],5,"KAN [1,5,1]"), ([1,3,3,1],8,"KAN [1,3,3,1] G=8")]:
        k_ = KANRegressor(width=w, grid=g, k=3)
        k_.fit(ds, steps=50, lamb=0.001)
        e = k_.evaluate(ds)
        e.update({"model":nm,"dataset":dn,"params":k_.count_parameters(),"time":k_.train_time})
        results_1d.append(e)
        print(f"  {nm}: R2={e['r2']:.4f}, MSE={e['mse']:.6f}, params={e['params']}")
    for h, nm in [([32,16],"MLP [32,16]"), ([64,32,16],"MLP [64,32,16]")]:
        m_ = MLPRegressor(hidden_layers=h, epochs=300); m_.fit(ds)
        e = m_.evaluate(ds)
        e.update({"model":nm,"dataset":dn,"params":m_.count_parameters(),"time":m_.train_time})
        results_1d.append(e)
        print(f"  {nm}: R2={e['r2']:.4f}, MSE={e['mse']:.6f}, params={e['params']}")

In [ ]:
df_1d = pd.DataFrame(results_1d)[["dataset","model","params","r2","mse","rmse","mae","time"]]
df_1d.columns = ["Dataset","Model","Params","R2","MSE","RMSE","MAE","Czas[s]"]
print(df_1d.to_string(index=False, float_format="%.4f"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, (nm, ds) in enumerate(datasets_1d.items()):
    ax = axes[idx]
    ai = torch.cat([ds["train_input"], ds["test_input"]]).cpu().numpy()
    al = torch.cat([ds["train_label"], ds["test_label"]]).cpu().numpy().flatten()
    si = np.argsort(ai.flatten()); xs, ys = ai.flatten()[si], al[si]
    ax.plot(xs, ys, "k-", label="Truth", linewidth=2, alpha=0.7)
    k_ = KANRegressor(width=[1,5,1], grid=5, k=3); k_.fit(ds, steps=50, lamb=0.001)
    ax.plot(xs, k_.predict(ai).flatten()[si], "--", label="KAN", linewidth=2)
    m_ = MLPRegressor(hidden_layers=[64,32], epochs=300); m_.fit(ds)
    ax.plot(xs, m_.predict(ai).flatten()[si], "-.", label="MLP", linewidth=2)
    ax.set_title(nm); ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(fontsize=9)
plt.suptitle("KAN vs MLP - funkcje 1D", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("./figures/predictions_1d.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Eksperymenty 2D: $f(x_1, x_2) = e^{\sin(\pi x_1) + x_2^2}$

In [ ]:
ds2d = KANDatasets.create_2d_dataset(n_samples=1000, noise=0.1)
results_2d = []
for w,g,nm in [([2,5,1],5,"KAN [2,5,1]"),([2,5,3,1],5,"KAN [2,5,3,1]"),([2,5,5,1],8,"KAN [2,5,5,1] G=8")]:
    k_ = KANRegressor(width=w, grid=g, k=3); k_.fit(ds2d, steps=50, lamb=0.001)
    e = k_.evaluate(ds2d)
    e.update({"model":nm,"params":k_.count_parameters(),"time":k_.train_time})
    results_2d.append(e)
    print(f"  {nm}: R2={e['r2']:.4f}, params={e['params']}")
for h,nm in [([32,16],"MLP [32,16]"),([64,32],"MLP [64,32]"),([128,64,32],"MLP [128,64,32]")]:
    m_ = MLPRegressor(hidden_layers=h, epochs=500); m_.fit(ds2d)
    e = m_.evaluate(ds2d)
    e.update({"model":nm,"params":m_.count_parameters(),"time":m_.train_time})
    results_2d.append(e)
    print(f"  {nm}: R2={e['r2']:.4f}, params={e['params']}")
print("\n"+pd.DataFrame(results_2d)[["model","params","r2","mse","time"]].to_string(index=False,float_format="%.4f"))

## 8. Diabetes (442 probek, 10 cech)

In [ ]:
ds_d = KANDatasets.create_diabetes_dataset()
results_d = []
for w,g,nm in [([10,8,1],3,"KAN [10,8,1]"),([10,8,4,1],5,"KAN [10,8,4,1]"),([10,16,8,1],5,"KAN [10,16,8,1]")]:
    k_ = KANRegressor(width=w, grid=g, k=3)
    k_.fit(ds_d, steps=80, lamb=0.01, lamb_entropy=2.0)
    e = k_.evaluate(ds_d)
    e.update({"model":nm,"params":k_.count_parameters(),"time":k_.train_time})
    results_d.append(e)
for h,nm in [([32,16],"MLP [32,16]"),([64,32,16],"MLP [64,32,16]"),([128,64,32],"MLP [128,64,32]")]:
    m_ = MLPRegressor(hidden_layers=h, epochs=500); m_.fit(ds_d)
    e = m_.evaluate(ds_d)
    e.update({"model":nm,"params":m_.count_parameters(),"time":m_.train_time})
    results_d.append(e)
df_d = pd.DataFrame(results_d)[["model","params","r2","mse","rmse","mae","time"]]
df_d.columns=["Model","Params","R2","MSE","RMSE","MAE","Czas[s]"]
print("Diabetes:")
print(df_d.to_string(index=False, float_format="%.4f"))

## 9. Grid refinement KAN

$MSE \sim O(G^{-(k+1)})$ -- unikalna cecha KAN, niedostepna w MLP.
Wzorzec z `tutorials/Example/Example_1_function_fitting`: iteracyjne `model.refine(G)`.

In [ ]:
ds_gr = KANDatasets.create_sinusoidal_dataset(n_samples=500, noise=0.0)
grids = [3, 5, 8, 10, 15, 20, 30]
gr_res = []
mdl = None
for i, g in enumerate(grids):
    if i == 0:
        mdl = KAN(width=[1,5,1], grid=g, k=3, seed=SEED, device=DEVICE)
    else:
        mdl = mdl.refine(g)
    mdl.fit(ds_gr, opt="LBFGS", steps=50, lamb=0.001)
    mdl.eval()
    with torch.no_grad():
        yp = mdl(ds_gr["test_input"]).cpu().numpy().flatten()
    yt = ds_gr["test_label"].cpu().numpy().flatten()
    mse = mean_squared_error(yt, yp)
    np_ = sum(p.numel() for p in mdl.parameters())
    gr_res.append({"grid": g, "mse": mse, "params": np_})
    print(f"  Grid={g:>3}: MSE={mse:.2e}, Params={np_}")

df_gr = pd.DataFrame(gr_res)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
a1.loglog(df_gr["grid"], df_gr["mse"], "o-", markersize=8)
a1.set_xlabel("Grid"); a1.set_ylabel("MSE"); a1.set_title("Zbieznosc KAN"); a1.grid(True, alpha=0.3)
a2.loglog(df_gr["params"], df_gr["mse"], "s-", markersize=8)
a2.set_xlabel("Params"); a2.set_ylabel("MSE"); a2.set_title("Efektywnosc param."); a2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("./figures/grid_refinement.png", dpi=150, bbox_inches="tight")
plt.show()
slope, _ = np.polyfit(np.log(df_gr["grid"].values), np.log(df_gr["mse"].values+1e-15), 1)
print(f"\nEmpiryczne: MSE ~ G^({slope:.2f}), Teoria: G^(-4) dla k=3")

## 10. Analiza czasu treningu i efektywnosci

In [ ]:
all_res = results_1d + results_2d + results_d
df_all = pd.DataFrame(all_res)
df_all["type"] = df_all["model"].apply(lambda x: "KAN" if "KAN" in x else "MLP")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for t, gr in df_all.groupby("type"):
    mk = "o" if t=="KAN" else "s"
    axes[0].scatter(gr["params"], gr["r2"], label=t, marker=mk, s=80, alpha=0.7)
    axes[1].scatter(gr["params"], gr["time"], label=t, marker=mk, s=80, alpha=0.7)
    axes[2].scatter(gr["time"], gr["r2"], label=t, marker=mk, s=80, alpha=0.7)
axes[0].set_xlabel("Params"); axes[0].set_ylabel("R2"); axes[0].set_xscale("log")
axes[0].set_title("R2 vs Params"); axes[0].legend()
axes[1].set_xlabel("Params"); axes[1].set_ylabel("Czas[s]"); axes[1].set_xscale("log")
axes[1].set_title("Czas vs Params"); axes[1].legend()
axes[2].set_xlabel("Czas[s]"); axes[2].set_ylabel("R2")
axes[2].set_title("R2 vs Czas"); axes[2].legend()
plt.tight_layout()
plt.savefig("./figures/efficiency_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("\n"+"="*70+"\nPODSUMOWANIE\n"+"="*70)
sm = df_all.groupby("type").agg({"r2":["mean","std","max"],"mse":["mean","min"],
    "params":["mean","min","max"],"time":["mean","sum"]}).round(4)
print(sm)
ka = df_all[df_all["type"]=="KAN"][["r2","params","time"]].mean()
ml = df_all[df_all["type"]=="MLP"][["r2","params","time"]].mean()
print(f"\nSrednie R2:   KAN={ka['r2']:.4f}  MLP={ml['r2']:.4f}")
print(f"Srednie p.:   KAN={ka['params']:.0f}  MLP={ml['params']:.0f}")
print(f"Sredni czas:  KAN={ka['time']:.2f}s  MLP={ml['time']:.2f}s")

## 11. Interpretowalnosc KAN

KAN pozwala wizualizowac nauczone funkcje aktywacji na krawedziach.
Wzorce z `API_2_plotting`: `model.plot(beta=)`, metryki `forward_n`, `forward_u`, `backward`.

In [ ]:
ds_i = KANDatasets.create_sinusoidal_dataset(n_samples=500, noise=0.0)
m_i = KAN(width=[1,3,1], grid=10, k=3, seed=SEED, device=DEVICE)
m_i.fit(ds_i, opt="LBFGS", steps=100, lamb=0.01)

print("Struktura KAN (domyslna metryka backward):")
m_i.plot()
plt.savefig("./figures/kan_structure.png", dpi=150, bbox_inches="tight")
plt.show()

# Porownanie metryk wizualizacji
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for idx, metric in enumerate(["backward", "forward_n", "forward_u"]):
    plt.sca(axes[idx])
    m_i.plot(metric=metric)
    axes[idx].set_title(f"Metryka: {metric}")
plt.suptitle("Porownanie metryk wizualizacji KAN", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("./figures/kan_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Pruning sieci KAN

Pruning to unikalna cecha KAN -- mozna usuwac zbedne neurony i krawedzie,
redukujac model bez znacznej utraty dokladnosci.
Wzorce z `tutorials/API_demo/API_7_pruning`.

In [ ]:
# Trening wiekszej sieci, nastepnie pruning
print("=" * 60)
print("PRUNING KAN")
print("=" * 60)

f_prune = lambda x: torch.exp(torch.sin(torch.pi*x[:,[0]]) + x[:,[1]]**2)
ds_prune = create_dataset(f_prune, n_var=2, device=DEVICE)

# Celowo wieksza siec
model_big = KAN(width=[2,5,1], grid=5, k=3, seed=SEED, device=DEVICE)
model_big.fit(ds_prune, opt="LBFGS", steps=50, lamb=0.01)

np_before = sum(p.numel() for p in model_big.parameters())
model_big.eval()
with torch.no_grad():
    yp = model_big(ds_prune["test_input"]).cpu().numpy().flatten()
yt = ds_prune["test_label"].cpu().numpy().flatten()
r2_before = r2_score(yt, yp)
print(f"Przed pruningiem: params={np_before}, R2={r2_before:.4f}")

print("\nStruktura przed pruningiem:")
model_big.plot()
plt.savefig("./figures/kan_before_prune.png", dpi=150, bbox_inches="tight")
plt.show()

# Pruning
model_pruned = model_big.prune()
model_pruned.fit(ds_prune, opt="LBFGS", steps=30)

np_after = sum(p.numel() for p in model_pruned.parameters())
model_pruned.eval()
with torch.no_grad():
    yp2 = model_pruned(ds_prune["test_input"]).cpu().numpy().flatten()
r2_after = r2_score(yt, yp2)
print(f"\nPo pruningu: params={np_after}, R2={r2_after:.4f}")
print(f"Redukcja parametrow: {100*(1-np_after/np_before):.1f}%")
print(f"Zmiana R2: {r2_after-r2_before:+.4f}")

print("\nStruktura po pruningu:")
model_pruned.plot()
plt.savefig("./figures/kan_after_prune.png", dpi=150, bbox_inches="tight")
plt.show()

## 13. Symbolic fitting - dopasowanie symboliczne

KAN potrafi odkryc symboliczna formule nauczonej funkcji.
Wykorzystuje `suggest_symbolic()` i `fix_symbolic()` do konwersji
nauczonych splajnow na znane funkcje matematyczne.
Wzorce z `tutorials/Example/Example_5_special_functions` i `Example_9_singularity`.

In [ ]:
# Prosta funkcja: f(x,y) = sin(pi*x) + y^2
print("=" * 60)
print("SYMBOLIC FITTING")
print("=" * 60)

f_sym = lambda x: torch.sin(torch.pi*x[:,[0]]) + x[:,[1]]**2
ds_sym = create_dataset(f_sym, n_var=2, device=DEVICE)

model_sym = KAN(width=[2,5,1], grid=10, k=3, seed=SEED, device=DEVICE)
model_sym.fit(ds_sym, opt="LBFGS", steps=50, lamb=0.01)

print("\nStruktura przed symbolic fitting:")
model_sym.plot()
plt.savefig("./figures/kan_symbolic_before.png", dpi=150, bbox_inches="tight")
plt.show()

# Pruning aby zostawic istotne krawedzie
model_sym2 = model_sym.prune()
model_sym2.fit(ds_sym, opt="LBFGS", steps=30)

print("\nPo pruningu:")
model_sym2.plot()
plt.show()

# Sugestie symboliczne dla kazdej krawedzi
print("\nSugestie symboliczne:")
w = [x[0] if isinstance(x, list) else x for x in model_sym2.width]
n_layers = len(w) - 1
for l in range(n_layers):
    for i in range(w[l]):
        for j in range(w[l+1]):
            try:
                result = model_sym2.suggest_symbolic(l, i, j)
                if result is not None:
                    print(f"  Layer {l}, edge ({i},{j}): {result}")
            except:
                pass

# Proba auto_symbolic
try:
    model_sym2.auto_symbolic()
    model_sym2.fit(ds_sym, opt="LBFGS", steps=30)
    formula = model_sym2.symbolic_formula()
    print(f"\nOdkryta formula: {formula[0][0]}")
    print("\nStruktura po symbolic fitting (czerwone = symboliczne):")
    model_sym2.plot()
    plt.savefig("./figures/kan_symbolic_after.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"auto_symbolic: {e}")

## 14. Feature attribution - waznosc cech

KAN automatycznie ocenia waznosc cech wejsciowych (`feature_score`).
Przydatne dla danych wielowymiarowych (Diabetes).
Wzorce z `tutorials/Interp/Interp_4_feature_attribution`.

In [ ]:
# Feature attribution na zbiorze Diabetes
print("=" * 60)
print("FEATURE ATTRIBUTION - DIABETES")
print("=" * 60)

ds_fa = KANDatasets.create_diabetes_dataset()
model_fa = KAN(width=[10,8,4,1], grid=5, k=3, seed=SEED, device=DEVICE)
model_fa.fit(ds_fa, opt="LBFGS", steps=80, lamb=0.01, lamb_entropy=2.0)

# Feature scores
scores = model_fa.feature_score.cpu().detach().numpy()

feature_names = ["age","sex","bmi","bp","s1","s2","s3","s4","s5","s6"]
df_feat = pd.DataFrame({"feature": feature_names, "score": scores}).sort_values("score", ascending=False)

print("\nWaznosc cech (feature_score):") 
print(df_feat.to_string(index=False, float_format="%.4f"))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_feat["feature"], df_feat["score"])
ax.set_xlabel("Feature Score"); ax.set_title("KAN Feature Attribution - Diabetes")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("./figures/feature_attribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Pruning inputs
try:
    model_pruned_input = model_fa.prune_input(threshold=1e-2)
    remaining = model_pruned_input.width[0]
    print(f"\nPo prune_input: {remaining}/{len(feature_names)} cech pozostalo")
    model_pruned_input.plot()
    plt.savefig("./figures/kan_pruned_inputs.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"prune_input: {e}")

## 15. Metryki regularyzacji (reg_metric)

KAN oferuje rozne metryki regularyzacji L1 (z `tutorials/API_demo/API_8_regularization`):
- `edge_forward_spline_n` (domyslna) - znormalizowana norma krawedzi
- `edge_forward_spline_u` - nieznormalizowana
- `edge_forward_sum` - suma aktywacji
- `edge_backward` - wsteczna propagacja waznosci

In [ ]:
print("=" * 60)
print("POROWNANIE reg_metric")
print("=" * 60)

ds_rm = KANDatasets.create_2d_dataset(n_samples=500, noise=0.05)

reg_metrics = [
    "edge_forward_spline_n",
    "edge_forward_spline_u",
    "edge_forward_sum",
    "edge_backward",
]

rm_results = []
for rm in reg_metrics:
    try:
        m = KANRegressor(width=[2,5,1], grid=5, k=3)
        m.fit(ds_rm, steps=50, lamb=0.01, reg_metric=rm)
        ev = m.evaluate(ds_rm)
        ev["reg_metric"] = rm
        ev["params"] = m.count_parameters()
        rm_results.append(ev)
        print(f"  {rm:<30} R2={ev['r2']:.4f}, MSE={ev['mse']:.6f}")
    except Exception as e:
        print(f"  {rm}: {e}")

if rm_results:
    df_rm = pd.DataFrame(rm_results)[["reg_metric","r2","mse","rmse"]]
    print("\n" + df_rm.to_string(index=False, float_format="%.4f"))

    # Wizualizacja modeli z rozna regularyzacja
    fig, axes = plt.subplots(1, len(reg_metrics), figsize=(5*len(reg_metrics), 4))
    for idx, rm in enumerate(reg_metrics):
        try:
            m = KAN(width=[2,5,1], grid=5, k=3, seed=SEED, device=DEVICE)
            m.fit(ds_rm, opt="LBFGS", steps=50, lamb=0.01, reg_metric=rm)
            plt.sca(axes[idx])
            m.plot()
            axes[idx].set_title(rm.replace("edge_",""), fontsize=9)
        except:
            axes[idx].set_title(f"{rm}\n(error)", fontsize=9)
    plt.suptitle("Wplyw reg_metric na strukture KAN", fontsize=14, y=1.05)
    plt.tight_layout()
    plt.savefig("./figures/reg_metric_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

## 16. Odpornosc na szum i efektywnosc probkowa

In [ ]:
noise_levels = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
nr = []
for noise in noise_levels:
    ds = KANDatasets.create_sinusoidal_dataset(n_samples=500, noise=noise)
    k_ = KANRegressor(width=[1,5,1], grid=5, k=3); k_.fit(ds, steps=50, lamb=0.01)
    m_ = MLPRegressor(hidden_layers=[64,32], epochs=300); m_.fit(ds)
    nr.append({"noise":noise,"KAN":k_.evaluate(ds)["r2"],"MLP":m_.evaluate(ds)["r2"]})
    print(f"  noise={noise:.2f}: KAN={nr[-1]['KAN']:.4f}, MLP={nr[-1]['MLP']:.4f}")
df_n = pd.DataFrame(nr)
plt.figure(figsize=(10, 6))
plt.plot(df_n["noise"], df_n["KAN"], "o-", label="KAN", markersize=8)
plt.plot(df_n["noise"], df_n["MLP"], "s-", label="MLP", markersize=8)
plt.xlabel("Szum"); plt.ylabel("R2"); plt.title("Odpornosc na szum")
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig("./figures/noise_robustness.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sizes = [50, 100, 200, 300, 500, 800, 1000]
sr = []
for n in sizes:
    ds = KANDatasets.create_sinusoidal_dataset(n_samples=n, noise=0.1)
    k_ = KANRegressor(width=[1,5,1], grid=5, k=3); k_.fit(ds, steps=50, lamb=0.01)
    m_ = MLPRegressor(hidden_layers=[64,32], epochs=300); m_.fit(ds)
    sr.append({"n":n,"KAN":k_.evaluate(ds)["r2"],"MLP":m_.evaluate(ds)["r2"]})
    print(f"  n={n:>4}: KAN={sr[-1]['KAN']:.4f}, MLP={sr[-1]['MLP']:.4f}")
df_s = pd.DataFrame(sr)
plt.figure(figsize=(10, 6))
plt.plot(df_s["n"], df_s["KAN"], "o-", label="KAN", markersize=8)
plt.plot(df_s["n"], df_s["MLP"], "s-", label="MLP", markersize=8)
plt.xlabel("Probki"); plt.ylabel("R2"); plt.title("Efektywnosc probkowa")
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig("./figures/sample_efficiency.png", dpi=150, bbox_inches="tight")
plt.show()

## 17. Podsumowanie

1. **Parametry:** KAN mniejsza architektura, porownywalna dokladnosc
2. **HP tuning:** KAN wrazliwszy na grid/lamb niz MLP na lr/hidden
3. **Grid refinement:** unikalna cecha KAN -- progresywne zwiekszanie rozdzielczosci
4. **Interpretowalnosc:** krawedzie KAN = wizualizowalne funkcje aktywacji
5. **Pruning:** KAN pozwala redukowac siec bez utraty dokladnosci
6. **Symbolic fitting:** KAN moze odkryc formule symboliczna
7. **Feature attribution:** automatyczna ocena waznosci cech
8. **Regularyzacja:** 4 metryki reg_metric do kontroli rzadkosci
9. **Ograniczenia:** KAN wolniejszy, gorzej skaluje sie wymiarowo
10. **Rekomendacje:** KAN dla interpretowalnosci/niskich dim; MLP dla produkcji

In [ ]:
print("="*80+"\nFINALNA TABELA\n"+"="*80)
comp = pd.DataFrame({
    "Kryterium": ["Teoria","Params/edge","Aktywacja","Interpretowalnosc",
        "Grid refine","Pruning","Symbolic","Feature attr.",
        "Optimizer","Szybkosc","Skalowanie","Sample eff.","HP sens.","Dojrzalosc"],
    "KAN": ["Kolmogorow-Arnold","G+k+1","Nauczana (B-spline)","Wysoka","Tak",
        "Tak (prune)","Tak (auto_symbolic)","Tak (feature_score)",
        "LBFGS","Wolniejszy","Gorsze","Lepsza","Wysoka","2024+"],
    "MLP": ["Aproks. uniwersalna","n_in+1","Stala (ReLU/tanh)","Niska","Nie",
        "Magnitude pruning","Nie","SHAP/LIME (zewn.)",
        "Adam/SGD","Szybszy","Lepsze","Wiecej danych","Umiarkowana","Dekady"]
})
print(comp.to_string(index=False))
print(f"\nDEVICE={DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif DEVICE == "mps":
    print("Apple Silicon GPU (MPS)")
else:
    print("CPU")